In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
import asyncio

In [9]:
from openai import AsyncOpenAI, OpenAI
sync_client = OpenAI()
client = AsyncOpenAI()

In [10]:
def llm_call(prompt: str, model: str = 'gpt-4o-mini') -> str:
    messages = []
    messages.append({'role':'user','content':prompt})
    chat_completion = sync_client.chat.completions.create(
        model = model,
        messages=messages
    )
    return chat_completion.choices[0].message.content

In [11]:
async def llm_call_async(prompt: str, model: str = 'gpt-4o-mini') -> str:
    messages = []
    messages.append({'role':'user','content':prompt})
    chat_completion = await client.chat.completions.create(
        model = model,
        messages=messages
    )
    print(model, "완료")
    
    return chat_completion.choices[0].message.content

In [12]:
async def run_llm_parallel(prompt_list):
    tasks = [llm_call_async(prompt) for prompt in prompt_list]
    responses = []
    for task in asyncio.as_completed(tasks):
        result = await task
        responses.append(result)
    return responses

In [13]:
user_query = 'AI는 미래 일자리에 어떤 영향을 미칠까?'

In [14]:
orchestrator_prompt= f"""
다음 사용자 질문을 분석하고, 이를 3개의 관련된 하위 질문으로 분해하십시오:

다음 형식으로 응답을 제공하십시오:

{{
    "analysis": "사용자 질문에 대한 이해를 상세히 설명하고, 작성한 하위 질문들의 근거를 설명하십시오.",
    "subtasks": [
        {{
            "description": "이 하위 질문의 초점과 의도를 설명하십시오.",
            "sub_question": "질문 1"
        }},
        {{
            "description": "이 하위 질문의 초점과 의도를 설명하십시오.",
            "sub_question": "질문 2"
        }}
        // 필요에 따라 추가 하위 질문 포함
    ]
}}
최대 3개의 하위 질문을 생성하세요

사용자 질문: {user_query}"""

In [16]:
orchestrator_reponse = llm_call(orchestrator_prompt, model = 'gpt-4o')

In [17]:
orchestrator_reponse

'```json\n{\n    "analysis": "사용자 질문은 인공지능(AI)이 미래의 고용 시장 및 일자리 환경에 어떤 변화를 가져올지에 대한 관심을 나타냅니다. 이는 기술 발전이 경제 및 사회 구조에 미치는 영향을 이해하려는 시도로 볼 수 있습니다. 본 질문은 AI가 노동 시장을 어떻게 변화시킬지, 이에 따라 산업 및 개인은 어떻게 대응해야 할지를 탐구하는 것을 목표로 합니다.",\n    "subtasks": [\n        {\n            "description": "AI 기술의 발전이 다양한 산업 분야에서의 일자리 요구 사항을 어떻게 변화시킬지 탐구하려는 의도로, 특정 직군이나 산업이 어떻게 영향을 받을지를 분석하려고 합니다.",\n            "sub_question": "AI 기술 발전이 특정 산업의 일자리 구조에 어떤 변화를 가져올까요?"\n        },\n        {\n            "description": "AI의 도입으로 인해 필요 없어지거나 새롭게 생겨나는 직업의 유형을 이해하기 위한 질문입니다. 이를 통해 직업 시장의 변화 양상을 예측하려고 합니다.",\n            "sub_question": "AI의 도입으로 인해 없어질 직업과 새롭게 생겨날 직업에는 무엇이 있을까요?"\n        },\n        {\n            "description": "AI로 인해 변화하는 일자리 환경에 적응하기 위해 개인과 기업이 준비해야 할 사항을 이해하려는 질문입니다. 이는 미래에 요구되는 기술 및 역량이 무엇인지를 파악하려는 목표를 가지고 있습니다.",\n            "sub_question": "미래의 직업 환경에 대비해 개인과 기업이 준비해야 할 사항은 무엇일까요?"\n        }\n    ]\n}\n```'

In [18]:
import json

In [19]:
response_json = json.loads(orchestrator_reponse.replace('```json', ' ').replace('```', ' '))

In [21]:
response_json

{'analysis': '사용자 질문은 인공지능(AI)이 미래의 고용 시장 및 일자리 환경에 어떤 변화를 가져올지에 대한 관심을 나타냅니다. 이는 기술 발전이 경제 및 사회 구조에 미치는 영향을 이해하려는 시도로 볼 수 있습니다. 본 질문은 AI가 노동 시장을 어떻게 변화시킬지, 이에 따라 산업 및 개인은 어떻게 대응해야 할지를 탐구하는 것을 목표로 합니다.',
 'subtasks': [{'description': 'AI 기술의 발전이 다양한 산업 분야에서의 일자리 요구 사항을 어떻게 변화시킬지 탐구하려는 의도로, 특정 직군이나 산업이 어떻게 영향을 받을지를 분석하려고 합니다.',
   'sub_question': 'AI 기술 발전이 특정 산업의 일자리 구조에 어떤 변화를 가져올까요?'},
  {'description': 'AI의 도입으로 인해 필요 없어지거나 새롭게 생겨나는 직업의 유형을 이해하기 위한 질문입니다. 이를 통해 직업 시장의 변화 양상을 예측하려고 합니다.',
   'sub_question': 'AI의 도입으로 인해 없어질 직업과 새롭게 생겨날 직업에는 무엇이 있을까요?'},
  {'description': 'AI로 인해 변화하는 일자리 환경에 적응하기 위해 개인과 기업이 준비해야 할 사항을 이해하려는 질문입니다. 이는 미래에 요구되는 기술 및 역량이 무엇인지를 파악하려는 목표를 가지고 있습니다.',
   'sub_question': '미래의 직업 환경에 대비해 개인과 기업이 준비해야 할 사항은 무엇일까요?'}]}

In [33]:
analysis = response_json.get("analysis", "")
subtasks = response_json.get("subtasks", [])

In [34]:
analysis

'사용자 질문은 인공지능(AI)이 미래의 고용 시장 및 일자리 환경에 어떤 변화를 가져올지에 대한 관심을 나타냅니다. 이는 기술 발전이 경제 및 사회 구조에 미치는 영향을 이해하려는 시도로 볼 수 있습니다. 본 질문은 AI가 노동 시장을 어떻게 변화시킬지, 이에 따라 산업 및 개인은 어떻게 대응해야 할지를 탐구하는 것을 목표로 합니다.'

In [36]:
subtasks

[{'description': 'AI 기술의 발전이 다양한 산업 분야에서의 일자리 요구 사항을 어떻게 변화시킬지 탐구하려는 의도로, 특정 직군이나 산업이 어떻게 영향을 받을지를 분석하려고 합니다.',
  'sub_question': 'AI 기술 발전이 특정 산업의 일자리 구조에 어떤 변화를 가져올까요?'},
 {'description': 'AI의 도입으로 인해 필요 없어지거나 새롭게 생겨나는 직업의 유형을 이해하기 위한 질문입니다. 이를 통해 직업 시장의 변화 양상을 예측하려고 합니다.',
  'sub_question': 'AI의 도입으로 인해 없어질 직업과 새롭게 생겨날 직업에는 무엇이 있을까요?'},
 {'description': 'AI로 인해 변화하는 일자리 환경에 적응하기 위해 개인과 기업이 준비해야 할 사항을 이해하려는 질문입니다. 이는 미래에 요구되는 기술 및 역량이 무엇인지를 파악하려는 목표를 가지고 있습니다.',
  'sub_question': '미래의 직업 환경에 대비해 개인과 기업이 준비해야 할 사항은 무엇일까요?'}]

In [37]:
def get_worker_prompt(user_query, sub_question, description):
    return f'''
    다음 사용자 질문에서 파생된 하우 질문을 다루는 작업을 맡았습니다.:
    원래 질문 : {user_query}
    하위 질문 : {sub_question}

    지침 : {description}
    
    하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요. '''

In [38]:
worker_prompts = [get_worker_prompt(user_query, task['sub_question'], task['description']) for task in subtasks]

In [41]:
for i in worker_prompts:
    print(i)
    print("=====")


    다음 사용자 질문에서 파생된 하우 질문을 다루는 작업을 맡았습니다.:
    원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?
    하위 질문 : AI 기술 발전이 특정 산업의 일자리 구조에 어떤 변화를 가져올까요?

    지침 : AI 기술의 발전이 다양한 산업 분야에서의 일자리 요구 사항을 어떻게 변화시킬지 탐구하려는 의도로, 특정 직군이나 산업이 어떻게 영향을 받을지를 분석하려고 합니다.

    하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요. 
=====

    다음 사용자 질문에서 파생된 하우 질문을 다루는 작업을 맡았습니다.:
    원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?
    하위 질문 : AI의 도입으로 인해 없어질 직업과 새롭게 생겨날 직업에는 무엇이 있을까요?

    지침 : AI의 도입으로 인해 필요 없어지거나 새롭게 생겨나는 직업의 유형을 이해하기 위한 질문입니다. 이를 통해 직업 시장의 변화 양상을 예측하려고 합니다.

    하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요. 
=====

    다음 사용자 질문에서 파생된 하우 질문을 다루는 작업을 맡았습니다.:
    원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?
    하위 질문 : 미래의 직업 환경에 대비해 개인과 기업이 준비해야 할 사항은 무엇일까요?

    지침 : AI로 인해 변화하는 일자리 환경에 적응하기 위해 개인과 기업이 준비해야 할 사항을 이해하려는 질문입니다. 이는 미래에 요구되는 기술 및 역량이 무엇인지를 파악하려는 목표를 가지고 있습니다.

    하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요. 
=====


In [42]:
worker_responses = await run_llm_parallel(worker_prompts)

gpt-4o-mini 완료
gpt-4o-mini 완료
gpt-4o-mini 완료


In [43]:
aggregator_prompt = f"""아래는 사용자의 원래 질문에 대해서 하위 질문을 나누고 응답한 결과입니다.
아래 질문 및 응답내용을 포함한 최종 응답을 제공해주세요.
요청사항
하위질문 응답내용이 최대한 포괄적이고 상세하게 포함되어야 합니다
사용자의 원래 질문:
{user_query}

하위 질문 및 응답:
"""

In [44]:
for i in range(len(subtasks)):
    aggregator_prompt += f"\n{i+1}, 하위 질문 : {subtasks[i]['sub_question']}"
    aggregator_prompt += f"\n      응답 : {worker_responses[i]}\n"

In [45]:
print(aggregator_prompt)

아래는 사용자의 원래 질문에 대해서 하위 질문을 나누고 응답한 결과입니다.
아래 질문 및 응답내용을 포함한 최종 응답을 제공해주세요.
요청사항
하위질문 응답내용이 최대한 포괄적이고 상세하게 포함되어야 합니다
사용자의 원래 질문:
AI는 미래 일자리에 어떤 영향을 미칠까?

하위 질문 및 응답:

1, 하위 질문 : AI 기술 발전이 특정 산업의 일자리 구조에 어떤 변화를 가져올까요?
      응답 : AI의 도입으로 인해 직업 시장은 급변하고 있으며, 기존의 직업이 사라지는 것과 동시에 새로운 직업이 생겨나는 현상이 나타나고 있습니다. 아래에서 이러한 변화의 구체적인 예시와 배경을 살펴보겠습니다.

### 없어질 직업

1. **단순 반복 작업 직종**:
   - **예시**: 공장 조립 라인 작업자, 데이터 입력원, 전화상담원
   - **이유**: AI와 자동화 기술이 발전함에 따라 단순한 반복 작업은 로봇이나 소프트웨어로 대체될 가능성이 높습니다. 이러한 직업은 규칙적이고 예측 가능한 작업으로 구성되어 있어 자동화에 적합합니다.

2. **고전적인 서비스 및 핀테크 관련 직종**:
   - **예시**: 은행의 고객 서비스 담당자, 보험 청구 처리 직원
   - **이유**: 챗봇과 AI 기반의 서비스 플랫폼이 고객의 문의를 신속하게 처리할 수 있어, 전통적인 고객 서비스 역할이 줄어들고 있습니다.

3. **기타 전통적 사무직**:
   - **예시**: 사무직 비서, 일부 회계 및 세무 처리직
   - **이유**: AI 도구가 업무 자동화, 일정 관리, 문서 정리 등의 기능을 지원함에 따라 이러한 역할이 축소되고 있습니다.

### 새롭게 생겨날 직업

1. **AI 전문 인력**:
   - **예시**: AI 데이터 과학자, 머신러닝 엔지니어, AI 윤리 전문가
   - **이유**: AI 기술의 발전으로 데이터 분석, 모델 개발, AI 시스템의 윤리적 적용을 평가하는 전문가들이 필요해지고 있습니다. 이러한 분야에서는 고급 기술과 관련된 전문성이

In [46]:
final_response = llm_call(aggregator_prompt, model='gpt-5.2')

In [47]:
print(final_response)

AI는 미래 일자리에 광범위한 영향을 미치며, 핵심은 **“일자리의 소멸 vs. 창출”이 동시에 진행되고**, 산업별로 **업무 내용·필요 역량·고용 형태**까지 함께 재편된다는 점입니다. 아래는 제시된 하위 질문들의 응답 내용을 바탕으로, 미래 일자리 변화 양상을 포괄적으로 정리한 최종 답변입니다.

---

## 1) AI 기술 발전이 특정 산업의 일자리 구조에 가져올 변화

AI 도입이 확대되면서 직업 시장은 빠르게 바뀌고 있습니다. **규칙적·반복적·예측 가능한 업무는 자동화**되기 쉬운 반면, **AI를 설계·운영·검증하거나 AI를 활용해 성과를 내는 역할**은 새롭게 커집니다. 그 결과 “기존 직무의 축소/변형”과 “신규 직무의 등장”이 동시에 일어납니다.

### (1) 없어질(축소될 가능성이 큰) 직업/업무
1. **단순 반복 작업 직종**
   - 예시: 공장 조립 라인 작업자, 데이터 입력원, 전화상담원  
   - 이유: 반복적이고 규칙 기반인 업무는 로봇/소프트웨어 자동화에 적합해 대체 가능성이 큽니다.

2. **고전적인 서비스 및 핀테크 관련 직종(특히 처리/응대 중심)**
   - 예시: 은행 고객 서비스 담당자, 보험 청구 처리 직원  
   - 이유: 챗봇/AI 상담 및 자동 심사·처리 시스템이 문의 응대와 문서 처리의 상당 부분을 빠르게 대체합니다.

3. **기타 전통적 사무직(일부 기능 중심)**
   - 예시: 사무직 비서, 일부 회계 및 세무 처리직  
   - 이유: 일정 관리, 문서 정리, 회계 처리 등 정형 업무가 AI 도구로 자동화되며 역할이 축소될 수 있습니다.

### (2) 새롭게 생겨날(성장할 가능성이 큰) 직업/역할
1. **AI 전문 인력**
   - 예시: AI 데이터 과학자, 머신러닝 엔지니어, AI 윤리 전문가  
   - 이유: 모델 개발·운영뿐 아니라 공정성/책임성/안전성 등 “AI의 사회적 적용”을 다루는 전문성이 필요해집니다.

2. **AI와 함께 발전하는 직무(산업별 AI 활용 직무)**

## 5. Evaluation

In [86]:
evaluator_prompt = """
다음 요약을 평가하십시오:

평가기준
핵심 내용 포함 여부
원문의 핵심 개념과 논리적 흐름이 유지되어야 합니다.
불필요한 세부 사항은 줄이되, 핵심 정보가 누락되면 감점 요인입니다.
단어 선택이 다소 달라도, 주요 개념과 의미가 유지되면 PASS 가능합니다.
원문의 중요 개념 15% 이상이 빠졌다면 FAIL입니다.

정확성 & 의미 전달
요약이 원문의 의미를 왜곡하지 않고 정확하게 전달해야 합니다.
숫자, 인명, 날짜 등 객관적 정보가 틀리면 FAIL입니다.
문장이 다르게 표현되었더라도 원문의 의미를 유지하면 PASS 가능합니다.
논리적 비약이 크거나 잘못된 해석이 포함되면 FAIL입니다.

간결성 및 가독성
문장이 과하게 길거나 반복적이면 감점 요인입니다.
직역체 표현은 가독성을 해치지 않으면 허용 가능하지만, 지나치면 FAIL입니다.
일부 단어의 표현 방식이 달라도 자연스럽다면 PASS 가능합니다.
문장이 지나치게 어색해서 독해가 어렵다면 FAIL입니다.

문법 및 표현
맞춤법, 띄어쓰기 오류가 5개 이상이면 FAIL입니다.
사소한 문법 실수는 감점 요인이나, 의미 전달에 영향을 주면 FAIL입니다.
문장이 비문이거나 문맥상 어색한 표현이 많으면 FAIL입니다.

평가결과 응답예시
모든 기준이 충족되었으면 "평가결과 = PASS"를 출력하세요.
수정이 필요한 경우, 구체적인 문제점을 지적하고 반드시 개선 방향을 제시하세요.
중대한 오류가 있다면 "평가결과 = FAIL"을 출력하고, 반드시 주요 문제점을 설명하세요.

요약 결과 :"""

In [87]:
input_article = """
오픈AI가 몇 주 안에 새로운 모델인 'GPT-4.5'를 출시하며 분산돼 있던 생성형 인공지능(AI) 모델을 통합키로 했다. 추론용 모델인 'o' 시리즈를 정리하고 비(非)추론 모델인 'GPT' 시리즈로 합칠 예정이다.

13일 업계에 따르면 샘 알트먼 오픈AI 최고경영자(CEO)는 지난 12일 자신의 X(옛 트위터)에 'GPT-4.5'를 조만간 출시할 것이라고 밝혔다. 현 세대인 'GPT-4o'의 뒤를 잇는 마지막 '비추론 AI'로, 내부적으로는 '오라이언(Orion)'이라고 불렸다.

현재 챗GPT 이용자를 비롯한 오픈AI의 고객들은 'GPT-4o', 'o1', 'o3-미니', 'GPT-4' 등 모델들을 각자 선택해 활용하고 있다. 최신 모델은 'GPT-4'를 개선한 'GPT-4o'로, 'GPT-4'는 2023년 하반기, 'GPT-4o'는 2024년 상반기 출시됐다.

오픈AI는 'GPT-5'도 지난해 공개하려고 했으나, 예상보다 저조한 성과를 거둬 출시가 연기된 상태다. 이에 그간 연산 시간을 늘려 성능을 높인 'o'시리즈 추론 모델을 새롭게 내세웠다.

샘 알트먼 CEO는 "이후 공개될 'GPT-5'부터는 추론 모델인 'o'시리즈와 'GPT'를 통합하겠다"며 "모델과 제품라인이 복잡해졌음을 잘 알고 있고, 앞으로는 각 모델을 선택해 사용하기보다 그저 잘 작동하길 원한다"고 말했다.
    """

In [88]:
user_query = f''' 
당신의 목표는 주어진 기사를 요약하는 것입니다.
아래 주어진 기사 내용을 요약해주세요.
이전 시도의 요약과 피드백이 있다면, 이를 반영하여 개선되 요약을 작성을 하세요.

기사 내용:
{input_article}
'''

In [69]:
summary = llm_call(user_query, model='gpt-3.5-turbo')

In [70]:
summary

"오픈AI가 'GPT-4.5'를 곧 출시할 예정이며, 이를 통해 생성형 인공지능(AI) 모델을 통합할 것이다. 이전에 출시된 'GPT-4'와 'o' 시리즈 모델들을 정리하고 향후 'GPT-5'부터는 추론 모델과 생성형 모델을 통합하여 제공할 계획이다. 신규 모델 출시 일정은 지연되었으나, 연산 시간을 늘려 성능을 개선한 'o' 시리즈 모델을 통해 대응하고 있다. 이를 통해 사용자들은 더 간편하게 AI 모델을 선택하여 활용할 수 있을 것으로 기대된다."

In [71]:
final_evaluator_prompt = evaluator_prompt + summary

In [72]:
final_evaluator_prompt

'\n다음 요약을 평가하십시오:\n\n평가기준\n핵심 내용 포함 여부\n원문의 핵심 개념과 논리적 흐름이 유지되어야 합니다.\n불필요한 세부 사항은 줄이되, 핵심 정보가 누락되면 감점 요인입니다.\n단어 선택이 다소 달라도, 주요 개념과 의미가 유지되면 PASS 가능합니다.\n원문의 중요 개념 15% 이상이 빠졌다면 FAIL입니다.\n\n정확성 & 의미 전달\n요약이 원문의 의미를 왜곡하지 않고 정확하게 전달해야 합니다.\n숫자, 인명, 날짜 등 객관적 정보가 틀리면 FAIL입니다.\n문장이 다르게 표현되었더라도 원문의 의미를 유지하면 PASS 가능합니다.\n논리적 비약이 크거나 잘못된 해석이 포함되면 FAIL입니다.\n\n간결성 및 가독성\n문장이 과하게 길거나 반복적이면 감점 요인입니다.\n직역체 표현은 가독성을 해치지 않으면 허용 가능하지만, 지나치면 FAIL입니다.\n일부 단어의 표현 방식이 달라도 자연스럽다면 PASS 가능합니다.\n문장이 지나치게 어색해서 독해가 어렵다면 FAIL입니다.\n\n문법 및 표현\n맞춤법, 띄어쓰기 오류가 5개 이상이면 FAIL입니다.\n사소한 문법 실수는 감점 요인이나, 의미 전달에 영향을 주면 FAIL입니다.\n문장이 비문이거나 문맥상 어색한 표현이 많으면 FAIL입니다.\n\n평가결과 응답예시\n모든 기준이 충족되었으면 "평가결과 = PASS"를 출력하세요.\n수정이 필요한 경우, 구체적인 문제점을 지적하고 반드시 개선 방향을 제시하세요.\n중대한 오류가 있다면 "평가결과 = FAIL"을 출력하고, 반드시 주요 문제점을 설명하세요.\n\n요약 결과 :오픈AI가 \'GPT-4.5\'를 곧 출시할 예정이며, 이를 통해 생성형 인공지능(AI) 모델을 통합할 것이다. 이전에 출시된 \'GPT-4\'와 \'o\' 시리즈 모델들을 정리하고 향후 \'GPT-5\'부터는 추론 모델과 생성형 모델을 통합하여 제공할 계획이다. 신규 모델 출시 일정은 지연되었으나, 연산 시간을 늘려 성능을 개선한 \'o\' 시리즈 모델을 통해 대응하고 

In [73]:
evaluation_result = llm_call(final_evaluator_prompt, model = 'gpt-4o')

In [74]:
print(evaluation_result)

평가결과 = FAIL

1. 핵심 내용 포함 여부:
   - 원문이 제공되지 않아 정확한 비교는 어렵지만, 요약 내용만으로 판단할 때 핵심 정보 중복과 불명확함이 일부 있는 것으로 보입니다. 특히 'o' 시리즈 모델과 관련된 내용이 명확히 정리되지 않은 점이 문제입니다. 'o' 시리즈 모델에 대한 더 구체적인 정보를 제공해야 할 것입니다(예: 어떤 방식으로 성능이 개선되었는지 등).

2. 정확성 & 의미 전달:
   - 숫자, 인명, 날짜 등의 특정 정보가 포함되지 않아 이 부분의 평가를 하기가 어렵습니다. 하지만 "이전에는 'GPT-4'와 'o' 시리즈 모델들을 정리하고"라는 표현이 다소 불명확합니다.
   - '정리'라는 표현이 어떤 의미로 쓰였는지 불명확하며, 이로 인해 의미가 왜곡될 수 있습니다.

3. 간결성 및 가독성:
   - 전반적으로 간결성과 가독성은 좋으나, "이전 모델을 정리"한다는 부분이 반복되는데, 이를 간결하게 표현하는 방법을 고민해야 합니다. 

4. 문법 및 표현:
   - 현재 상태로는 맞춤법 및 문법 오류는 보이지 않습니다. 그러나 'o' 시리즈 모델의 관련 설명이 명확하지 않아 표현 개선이 필요합니다.

개선 방향:
- 'o' 시리즈 모델의 역할과 개선 사항에 대한 구체적 설명을 추가해야 합니다.
- "정리하고"라는 표현 대신 더 명확한 설명을 통해 오해의 소지를 줄여야 합니다. 
- 문장을 좀 더 명확하게 재구성하여 사용자가 정보를 쉽게 이해할 수 있도록 해야 합니다.


In [75]:
retries = 1
user_query += f"{retries}차 요약 결과 : \n\n{summary}\n\n"

In [76]:
user_query += f"{retries}차 요약 피드백 : \n\n {evaluation_result}"

In [77]:
print(user_query)

 
당신의 목표는 주어진 기사를 요약하는 것입니다.
아래 주어진 기사 내용을 요약해주세요.
이전 시도의 요약과 피드백이 있다면, 이를 반영하여 개선되 요약을 작성을 하세요.

기사 내용:

오픈AI가 몇 주 안에 새로운 모델인 'GPT-4.5'를 출시하며 분산돼 있던 생성형 인공지능(AI) 모델을 통합키로 했다. 추론용 모델인 'o' 시리즈를 정리하고 비(非)추론 모델인 'GPT' 시리즈로 합칠 예정이다.

13일 업계에 따르면 샘 알트먼 오픈AI 최고경영자(CEO)는 지난 12일 자신의 X(옛 트위터)에 'GPT-4.5'를 조만간 출시할 것이라고 밝혔다. 현 세대인 'GPT-4o'의 뒤를 잇는 마지막 '비추론 AI'로, 내부적으로는 '오라이언(Orion)'이라고 불렸다.

현재 챗GPT 이용자를 비롯한 오픈AI의 고객들은 'GPT-4o', 'o1', 'o3-미니', 'GPT-4' 등 모델들을 각자 선택해 활용하고 있다. 최신 모델은 'GPT-4'를 개선한 'GPT-4o'로, 'GPT-4'는 2023년 하반기, 'GPT-4o'는 2024년 상반기 출시됐다.

오픈AI는 'GPT-5'도 지난해 공개하려고 했으나, 예상보다 저조한 성과를 거둬 출시가 연기된 상태다. 이에 그간 연산 시간을 늘려 성능을 높인 'o'시리즈 추론 모델을 새롭게 내세웠다.

샘 알트먼 CEO는 "이후 공개될 'GPT-5'부터는 추론 모델인 'o'시리즈와 'GPT'를 통합하겠다"며 "모델과 제품라인이 복잡해졌음을 잘 알고 있고, 앞으로는 각 모델을 선택해 사용하기보다 그저 잘 작동하길 원한다"고 말했다.
    
1차 요약 결과 : 

오픈AI가 'GPT-4.5'를 곧 출시할 예정이며, 이를 통해 생성형 인공지능(AI) 모델을 통합할 것이다. 이전에 출시된 'GPT-4'와 'o' 시리즈 모델들을 정리하고 향후 'GPT-5'부터는 추론 모델과 생성형 모델을 통합하여 제공할 계획이다. 신규 모델 출시 일정은 지연되었으나, 연산 시간을 늘려 성능을 개선한 'o' 시리즈 모델을 통해 대응하고 있다

In [78]:
summary = llm_call(user_query, model='gpt-3.5-turbo')

In [79]:
summary

"2차 요약 결과:\n\n오픈AI가 'GPT-4.5'를 곧 출시할 예정이며, 'o' 시리즈 모델과 'GPT' 시리즈 모델을 통합하여 제공할 계획이다. 'o' 시리즈 모델은 연산 시간을 늘려 성능을 높이는 방식으로 개선되었으며, 'GPT-5'부터는 추론 모델과 생성형 모델을 통합하여 제공할 예정이다. 이를 통해 사용자들은 더 간편하게 AI 모델을 선택하여 활용할 수 있을 것으로 기대된다."

In [80]:
second_evaluator_prompt = final_evaluator_prompt + summary

In [81]:
evaluation__second_result = llm_call(second_evaluator_prompt, model = 'gpt-4o')

In [82]:
print(evaluation__second_result)

평가결과 = FAIL

주요 문제점:
1. **핵심 내용 포함 여부**: 원문에서 언급된 'GPT-4'의 위치와 역할이 명확히 전달되지 않았습니다. 첫 번째 요약에서는 'GPT-4'와 'o 시리즈' 모델의 관계와 기능 변화에 대한 설명이 있었으나, 두 번째 요약에서는 이러한 관계와 변화를 명시하지 않아 원문의 핵심 내용 일부가 누락되었습니다.
   
2. **정확성 & 의미 전달**: 첫 번째 요약은 '신규 모델 출시 일정은 지연되었으나'라는 중요한 정보를 포함하고 있었으나, 두 번째 요약에서는 이러한 정보가 빠져 있어 의미 전달이 누락되었습니다.

3. **문법 및 표현**: 첫 번째 요약은 문장이 과도하게 길고 반복적이며, 연산 시간 증가에 대한 설명 부분이 다소 불분명합니다. 두 번째 요약에서도 'o 시리즈' 모델과 'GPT' 시리즈 모델의 구체적인 통합 방식이 명확하지 않다는 점에서 표현 오류가 있습니다.

개선 방향:
- 두 번째 요약에 'GPT-4'의 역할과 중요성을 명확히 언급하고, 왜 'o 시리즈' 모델의 연산 시간 증가가 중요한지 설명을 추가해야 합니다.
- '신규 모델 출시 일정의 지연'이라는 주요 정보를 포함하여 의미 전달을 강화해야 합니다.
- 요약 문장을 좀 더 간결하고 명확하게 개선하여 가독성을 높일 필요가 있습니다.


In [ ]:
retries = 0
while True:
    
    summary = llm_call(user_query, model='gpt-3.5-turbo')
    final_evaluator_prompt = evaluator_prompt + summary
    evaluation_result = llm_call(final_evaluator_prompt, model = 'gpt-4o')
    breaktime = evaluation_result.splitlines()[0].split('평가결과 = ')[1]
    if breaktime == 'PASS':
        break

    retries += 1
    user_query += f"{retries}차 요약 결과 : \n\n{summary}\n\n"
    user_query += f"{retries}차 요약 피드백 : \n\n {evaluation_result}"

    if retries == 5:
        break


print(f"시도 횟수 : {retries+1}")
print(evaluation_result)

시도 횟수 : 3
평가결과 = PASS
